# Cento Nuptialis Stylometry with LambdaG

This notebook analyses the stylistic affinities of Ausonius's *Cento Nuptialis* using
LambdaG authorship verification (uncalibrated).

## Setup

Three model corpora are compared against the cento:
- **vergil_model**: Vergil, *Georgics* (4 books)
- **ausonius_model**: Ausonius hexameter works (7 works)
- **background_model**: Ovid (*Metamorphoses*), Lucan (*Bellum Civile*), Silius Italicus (*Punica*), Valerius Flaccus (*Argonautica*)

All corpora contain `*_sentences.quesplit.masked.txt` files produced by the
latin-masking pipeline.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_corpora import Sentence, TwoLevelCorpus
from sentence_corpora.lambdag import LambdaGAV

## Load Corpora

Each directory contains `*_sentences.quesplit.masked.txt` files. We parse the
filenames to extract `(author, work)` metadata and build `TwoLevelCorpus` objects.

In [3]:
def parse_filename(path: Path) -> tuple[str, str]:
    """Parse '<author>_<work>_sentences.quesplit.masked.txt' into (author, work)."""
    name = re.sub(r"_sentences\.quesplit\.masked\.txt$", "", path.name)
    parts = name.split("_", 1)
    if len(parts) != 2:
        raise ValueError(f"Expected '<author>_<work>' in filename, got: {path.name!r}")
    return parts[0], parts[1]


def load_corpus(data_dir: Path) -> TwoLevelCorpus:
    """Read all *_sentences.quesplit.masked.txt files and return a TwoLevelCorpus."""
    sentences = []
    for path in sorted(data_dir.glob("*_sentences.quesplit.masked.txt")):
        author, work = parse_filename(path)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(Sentence(text=line, metadata={"author": author, "work": work}))
    return TwoLevelCorpus(sentences, levels=["author", "work"])


# Load the three model corpora
vergil_corpus = load_corpus(Path("vergil_model"))
ausonius_corpus = load_corpus(Path("ausonius_model"))
background_corpus = load_corpus(Path("background_model"))

# Load the question text (cento)
cento_path = Path("question_text_processed/cento_nuptialis_sentences.quesplit.masked.txt")
cento_sentences = []
with open(cento_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            cento_sentences.append(Sentence(text=line, metadata={"author": "cento", "work": "nuptialis"}))
cento_corpus = TwoLevelCorpus(cento_sentences, levels=["author", "work"])

print(f"Vergil corpus:      {len(vergil_corpus)} sentences, {sum(len(s.text.split()) for s in vergil_corpus)} tokens")
print(f"Ausonius corpus:    {len(ausonius_corpus)} sentences, {sum(len(s.text.split()) for s in ausonius_corpus)} tokens")
print(f"Background corpus:  {len(background_corpus)} sentences, {sum(len(s.text.split()) for s in background_corpus)} tokens")
print(f"Cento (question):   {len(cento_corpus)} sentences, {sum(len(s.text.split()) for s in cento_corpus)} tokens")

Vergil corpus:      809 sentences, 17802 tokens
Ausonius corpus:    389 sentences, 8193 tokens
Background corpus:  15111 sentences, 306869 tokens
Cento (question):   58 sentences, 1079 tokens


In [7]:
vergil_corpus[:10]

[Sentence(text='quid VERB ADJ NOUN quo NOUN NOUN <EOL> VERB PROPN ...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='uos o ADJ NOUN <EOL> NOUN VERB NOUN quae VERB NOUN...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='PROPN et ADJ NOUN uestro si NOUN NOUN <EOL> ADJ AD...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='et uos ADJ VERB NOUN PROPN <EOL> <EOS>...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='VERB simul PROPN -que NOUN PROPN -que NOUN <EOL> N...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='tu -que o cui ADJ VERB <EOL> VERB NOUN ADJ NOUN VE...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='et NOUN NOUN cui ADJ PROPN <EOL> ter NUM ADJ VERB ...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sentence(text='ipse NOUN VERB ADJ NOUN -que ADJ <EOL> PROPN NOUN ...', metadata={'author': 'vergil', 'work': 'georgics_1'}),
 Sen

In [8]:
ausonius_corpus[:10]

[Sentence(text='NOUN in NOUN VERB quos NOUN PROPN <EOL> ADJ VERB u...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='quorum per NOUN ADJ NOUN VERB <EOL> VERB olim NOUN...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='omnia quae NOUN et NOUN ADJ ADJ <EOL> VERB ADJ VER...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='ADJ PROPN VERB NOUN NOUN <EOL> VERB et VERB VERB p...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='ADJ NOUN VERB NOUN VERB ADJ <EOL> VERB in ADJ PROP...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='NOUN VERB adhuc PROPN PROPN -que VERB <EOL> VERB e...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='VERB ADJ NOUN <EOL> NOUN ADJ ADJ de NOUN NOUN <EOL...', metadata={'author': 'ausonius', 'work': 'cupido_cruciatus'}),
 Sentence(text='tota quoque ADJ ADJ NOUN PROPN <EOL> NOUN NOUN

## Run LambdaG Authorship Verification (Uncalibrated)

For each candidate author, we run a single AV problem:
- **q_corpus**: the cento (question text)
- **kr_corpus**: the model corpus (known reference)
- **known_entity**: the author label in the model corpus

The `LambdaGAV.run_single_av_problem` method computes a raw score indicating
how similar the question text is to the known author. Higher scores = more similar.

In [11]:
# Run LambdaG AV for each candidate author
# The cento is small (~900 tokens), so we use conservative known/reference token counts
av = LambdaGAV()
rng = np.random.default_rng(42)

results = []

for candidate, candidate_corpus in [("vergil", vergil_corpus), ("ausonius", ausonius_corpus)]:

    # Combine candidate + background into a single KR corpus
    kr_sentences = list(candidate_corpus) + list(background_corpus)
    kr_corpus = TwoLevelCorpus(kr_sentences, levels=["author", "work"])    

    result = av.run_single_av_problem(
        known_entity=candidate,
        q_corpus=cento_corpus,
        kr_corpus=kr_corpus,
        known_tokens=8000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=rng,
    )
    
    results.append({
        "candidate": candidate,
        "score": result.get("score", 0.0),
        "corrected_score_sqrt": result.get("corrected_score_sqrt", 0.0),
        "corrected_score_hapax": result.get("corrected_score_hapax", 0.0),
        "n_unknown": result.get("unknown_sentences_used", 0),
        "unknown_tokens": result.get("unknown_tokens_used", 0),
        "n_known": result.get("known_sentences_used", 0),
        "n_reference": result.get("reference_sentences_used", 0),
        "error": result.get("error", None),
    })
    print(f"  {candidate}: score={results[-1]['score']:.4f}, tokens_q={results[-1]['unknown_tokens']}")

print("\nDone.")

  vergil: score=15.4482, tokens_q=1079
  ausonius: score=-1.0804, tokens_q=1079

Done.


## Results

In [16]:
df = pd.DataFrame(results)
df = df.sort_values("score", ascending=False)

# Calculate probabilities from both corrected LLRs using prior=0.5
PRIOR = 0.5
df["prob_sqrt"] = df["corrected_score_sqrt"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)
df["prob_hapax"] = df["corrected_score_hapax"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)

print("LambdaG Results (uncalibrated, prior=0.5):")
print("=" * 80)
for _, row in df.iterrows():
    print(f"  {row['candidate']:<12}  raw={row['score']:>8.4f}"
          f"  sqrt={row['corrected_score_sqrt']:>8.4f} → {row['prob_sqrt']:>6.1f}%"
          f"  hapax={row['corrected_score_hapax']:>8.4f} → {row['prob_hapax']:>6.1f}%"
          f"  (q_tokens={row['unknown_tokens']})")



LambdaG Results (uncalibrated, prior=0.5):
  vergil        raw= 15.4482  sqrt=  0.4703 →   61.5%  hapax=  0.8018 →   69.0%  (q_tokens=1079)
  ausonius      raw= -1.0804  sqrt= -0.0329 →   49.2%  hapax= -0.0561 →   48.6%  (q_tokens=1079)


## Aeneid vs Vergil (Georgics) — Baseline Check

In [ ]:
from typing import Any


def sample_corpus_to_token_count(corpus, target_tokens: int, rng: Any):
    """Sample sentences from corpus until we reach approximately target_tokens.

    Shuffles sentences and adds them until we exceed target_tokens.
    """
    sentences = list(corpus)
    total_tokens = sum(len(s.text.split()) for s in sentences)

    if total_tokens <= target_tokens:
        raise ValueError(
            f"Corpus has only {total_tokens} tokens, which is less than target {target_tokens}"
        )

    shuffled = list(sentences)
    rng.shuffle(shuffled)

    sampled = []
    current_tokens = 0
    for s in shuffled:
        sampled.append(s)
        current_tokens += len(s.text.split())
        if current_tokens >= target_tokens:
            break

    return type(corpus)(sampled, levels=corpus.levels)


# Load Aeneid corpus
aeneid_corpus = load_corpus(Path("aeneid_corpus"))
print(f"Aeneid corpus: {len(aeneid_corpus)} sentences, {sum(len(s.text.split()) for s in aeneid_corpus)} tokens")

In [38]:
import ray
from tqdm import tqdm

# Run N iterations: sample ~1079 tokens from Aeneid, compare against Vergil (Georgics) + background
TARGET_TOKENS = 1079
NUM_ITERATIONS = 50

# KR corpus = vergil (Georgics) + background
kr_sentences = list(vergil_corpus) + list(background_corpus)
kr_corpus = TwoLevelCorpus(kr_sentences, levels=["author", "work"])


@ray.remote
def run_aeneid_iteration(i: int) -> dict:
    """Run a single Aeneid vs Vergil AV iteration."""
    iter_rng = np.random.default_rng(42 + i)
    q_corpus = sample_corpus_to_token_count(aeneid_corpus, TARGET_TOKENS, iter_rng)
    actual_tokens = sum(len(s.text.split()) for s in q_corpus)

    av = LambdaGAV()
    result = av.run_single_av_problem(
        known_entity="vergil",
        q_corpus=q_corpus,
        kr_corpus=kr_corpus,
        known_tokens=8000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=iter_rng,
    )

    return {
        "iteration": i,
        "score": result.get("score", 0.0),
        "corrected_score_sqrt": result.get("corrected_score_sqrt", 0.0),
        "corrected_score_hapax": result.get("corrected_score_hapax", 0.0),
        "q_tokens": actual_tokens,
        "n_known": result.get("known_sentences_used", 0),
        "n_reference": result.get("reference_sentences_used", 0),
        "error": result.get("error", None),
    }


ray.init(ignore_reinit_error=True)
print(f"Ray initialized with {ray.available_resources().get('CPU', 4)} CPUs")

tasks = [run_aeneid_iteration.remote(i) for i in range(NUM_ITERATIONS)]

aeneid_results = []
for _ in tqdm(range(len(tasks)), desc="Aeneid AV"):
    ready, tasks = ray.wait(tasks, num_returns=1)
    result = ray.get(ready[0])
    aeneid_results.append(result)

ray.shutdown()
print("\nDone.")

2026-07-23 10:50:32,384	INFO worker.py:2024 -- Started a local Ray instance.


Ray initialized with 10.0 CPUs


Aeneid AV: 100%|██████████| 50/50 [00:28<00:00,  1.74it/s]



Done.


In [39]:
df_aeneid = pd.DataFrame(aeneid_results)

PRIOR = 0.5
df_aeneid["prob_sqrt"] = df_aeneid["corrected_score_sqrt"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)
df_aeneid["prob_hapax"] = df_aeneid["corrected_score_hapax"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)

print(f"Aeneid vs Vergil (Georgics) — {NUM_ITERATIONS} iterations, ~{TARGET_TOKENS} tokens each, prior={PRIOR}")
print("=" * 100)
print(f"{'iter':>4}  {'raw':>8}  {'sqrt':>8}  {'prob_sqrt':>10}  {'hapax':>8}  {'prob_hapax':>10}  {'q_tok':>6}")
print("-" * 100)
for _, row in df_aeneid.iterrows():
    print(f"  {int(row['iteration']):>2}  {row['score']:>8.4f}  {row['corrected_score_sqrt']:>8.4f}"
          f"  {row['prob_sqrt']:>9.1f}%  {row['corrected_score_hapax']:>8.4f}"
          f"  {row['prob_hapax']:>9.1f}%  {int(row['q_tokens']):>6}")
print("-" * 100)
print(f"{'mean':>4}  {df_aeneid['score'].mean():>8.4f}  {df_aeneid['corrected_score_sqrt'].mean():>8.4f}"
      f"  {df_aeneid['prob_sqrt'].mean():>9.1f}%  {df_aeneid['corrected_score_hapax'].mean():>8.4f}"
      f"  {df_aeneid['prob_hapax'].mean():>9.1f}%")
print(f"{'std':>4}  {df_aeneid['score'].std():>8.4f}  {df_aeneid['corrected_score_sqrt'].std():>8.4f}"
      f"  {df_aeneid['prob_sqrt'].std():>9.1f}%  {df_aeneid['corrected_score_hapax'].std():>8.4f}"
      f"  {df_aeneid['prob_hapax'].std():>9.1f}%")

Aeneid vs Vergil (Georgics) — 50 iterations, ~1079 tokens each, prior=0.5
iter       raw      sqrt   prob_sqrt     hapax  prob_hapax   q_tok
----------------------------------------------------------------------------------------------------
   1   -3.0013   -0.0914       47.7%   -0.1975       45.1%    1079
   2  -47.8821   -1.4536       18.9%   -3.3098        3.5%    1085
   7    7.0878    0.2154       55.4%    0.5236       62.8%    1083
   9  -15.8186   -0.4780       38.3%   -1.0257       26.4%    1095
   0   -9.2652   -0.2796       43.1%   -0.6160       35.1%    1098
   8   73.8962    2.2444       90.4%    4.9082       99.3%    1084
   3   35.4415    1.0785       74.6%    2.5925       93.0%    1080
   5   72.7212    2.2108       90.1%    5.2424       99.5%    1082
   4   67.2103    2.0395       88.5%    4.5178       98.9%    1086
   6   54.8530    1.6561       84.0%    3.6502       97.5%    1097
  12    4.5624    0.1385       53.5%    0.2986       57.4%    1085
  10   68.4314    2.0

## Mosella vs Ausonius — Internal Baseline

In [36]:
# Build Mosella corpus (q) and Ausonius-minus-Mosella corpus (kr)
mosella_sentences = [s for s in ausonius_corpus if s.work == "mosella"]
ausonius_no_mosella_sentences = [s for s in ausonius_corpus if s.work != "mosella"]

mosella_corpus = TwoLevelCorpus(mosella_sentences, levels=["author", "work"])
ausonius_no_mosella = TwoLevelCorpus(ausonius_no_mosella_sentences, levels=["author", "work"])

# KR corpus = ausonius (without mosella) + background
kr_ausonius_sentences = list(ausonius_no_mosella) + list(background_corpus)
kr_ausonius_corpus = TwoLevelCorpus(kr_ausonius_sentences, levels=["author", "work"])

print(f"Mosella corpus: {len(mosella_corpus)} sentences, {sum(len(s.text.split()) for s in mosella_corpus)} tokens")
print(f"Ausonius (excl. Mosella): {len(ausonius_no_mosella)} sentences, {sum(len(s.text.split()) for s in ausonius_no_mosella)} tokens")
print(f"KR corpus (Ausonius excl. Mosella + background): {len(kr_ausonius_corpus)} sentences")


@ray.remote
def run_mosella_iteration(i: int) -> dict:
    """Run a single Mosella vs Ausonius AV iteration."""
    iter_rng = np.random.default_rng(100 + i)
    q_corpus = sample_corpus_to_token_count(mosella_corpus, TARGET_TOKENS, iter_rng)
    actual_tokens = sum(len(s.text.split()) for s in q_corpus)

    av = LambdaGAV()
    result = av.run_single_av_problem(
        known_entity="ausonius",
        q_corpus=q_corpus,
        kr_corpus=kr_ausonius_corpus,
        known_tokens=4000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=iter_rng,
    )

    return {
        "iteration": i,
        "score": result.get("score", 0.0),
        "corrected_score_sqrt": result.get("corrected_score_sqrt", 0.0),
        "corrected_score_hapax": result.get("corrected_score_hapax", 0.0),
        "q_tokens": actual_tokens,
        "n_known": result.get("known_sentences_used", 0),
        "n_reference": result.get("reference_sentences_used", 0),
        "error": result.get("error", None),
    }


ray.init(ignore_reinit_error=True)

tasks = [run_mosella_iteration.remote(i) for i in range(NUM_ITERATIONS)]

mosella_results = []
for _ in tqdm(range(len(tasks)), desc="Mosella AV"):
    ready, tasks = ray.wait(tasks, num_returns=1)
    result = ray.get(ready[0])
    mosella_results.append(result)

ray.shutdown()
print("\nDone.")

Mosella corpus: 163 sentences, 3688 tokens
Ausonius (excl. Mosella): 226 sentences, 4505 tokens
KR corpus (Ausonius excl. Mosella + background): 15337 sentences


2026-07-23 10:45:29,834	INFO worker.py:2024 -- Started a local Ray instance.
Mosella AV: 100%|██████████| 50/50 [00:15<00:00,  3.24it/s]



Done.


In [37]:
df_mosella = pd.DataFrame(mosella_results)

df_mosella["prob_sqrt"] = df_mosella["corrected_score_sqrt"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)
df_mosella["prob_hapax"] = df_mosella["corrected_score_hapax"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)

print(f"Mosella vs Ausonius — {NUM_ITERATIONS} iterations, ~{TARGET_TOKENS} tokens each, prior={PRIOR}")
print("=" * 100)
print(f"{'iter':>4}  {'raw':>8}  {'sqrt':>8}  {'prob_sqrt':>10}  {'hapax':>8}  {'prob_hapax':>10}  {'q_tok':>6}")
print("-" * 100)
for _, row in df_mosella.iterrows():
    print(f"  {int(row['iteration']):>2}  {row['score']:>8.4f}  {row['corrected_score_sqrt']:>8.4f}"
          f"  {row['prob_sqrt']:>9.1f}%  {row['corrected_score_hapax']:>8.4f}"
          f"  {row['prob_hapax']:>9.1f}%  {int(row['q_tokens']):>6}")
print("-" * 100)
print(f"{'mean':>4}  {df_mosella['score'].mean():>8.4f}  {df_mosella['corrected_score_sqrt'].mean():>8.4f}"
      f"  {df_mosella['prob_sqrt'].mean():>9.1f}%  {df_mosella['corrected_score_hapax'].mean():>8.4f}"
      f"  {df_mosella['prob_hapax'].mean():>9.1f}%")
print(f"{'std':>4}  {df_mosella['score'].std():>8.4f}  {df_mosella['corrected_score_sqrt'].std():>8.4f}"
      f"  {df_mosella['prob_sqrt'].std():>9.1f}%  {df_mosella['corrected_score_hapax'].std():>8.4f}"
      f"  {df_mosella['prob_hapax'].std():>9.1f}%")

Mosella vs Ausonius — 50 iterations, ~1079 tokens each, prior=0.5
iter       raw      sqrt   prob_sqrt     hapax  prob_hapax   q_tok
----------------------------------------------------------------------------------------------------
   1   22.0905    0.6642       66.0%    1.6977       84.5%    1106
   9  -24.7080   -0.7430       32.2%   -1.5415       17.6%    1106
   3   -2.9815   -0.0906       47.7%   -0.1516       46.2%    1082
   8   44.7071    1.3510       79.4%    2.1231       89.3%    1095
   0   95.3973    2.8895       94.7%    3.8509       97.9%    1090
   7   25.8863    0.7841       68.7%    1.4487       81.0%    1090
   5   52.1936    1.5802       82.9%    2.8704       94.6%    1091
   2   71.1263    2.1623       89.7%    5.1931       99.4%    1082
   6   99.0525    2.9798       95.2%    4.8406       99.2%    1105
   4   -0.7441   -0.0226       49.4%   -0.0376       49.1%    1087
  10   63.9598    1.9302       87.3%    4.0193       98.2%    1098
  13  100.9386    3.0630     